# ENSEMBLE LEARNING HOMEWORK: SPACESHIP TITANIC
**Mục tiêu:** Dự đoán hành khách có bị dịch chuyển sang không gian khác (Transported) hay không.
**Kỹ thuật áp dụng:** Feature Engineering và Ensemble Learning (Bagging, Boosting, Stacking/Voting).

In [44]:
%pip install xgboost lightgbm catboost scikit-learn pandas  

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Import các mô hình Ensemble
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings('ignore')

Note: you may need to restart the kernel to use updated packages.


## 1. Khám phá dữ liệu (EDA) và Trích xuất đặc trưng (Feature Engineering)

**Phân tích dữ liệu gốc:**
Bộ dữ liệu Spaceship Titanic chứa thông tin của hành khách bị lạc trong không gian. Qua quá trình phân tích (EDA), dữ liệu thô có rất nhiều cột chứa thông tin ẩn dạng chuỗi (String) và nhiều giá trị khuyết thiếu (Missing Values). Nếu đưa trực tiếp vào mô hình sẽ gây nhiễu và giảm độ chính xác.

**Kỹ thuật Feature Engineering áp dụng:**
Thay vì chỉ điền giá trị thiếu đơn thuần, chúng ta áp dụng kỹ thuật trích xuất đặc trưng để tạo ra các biến (features) mới có sức mạnh phân loại cao hơn:
*   **`Total_Spend` & `Spent_Money`:** Hành khách bị lạc có xu hướng chi tiêu khác biệt. Ta gom tổng các dịch vụ (RoomService, Spa, VRDeck...) lại. Nếu `Total_Spend == 0`, tạo cờ `Spent_Money = 0`.
*   **Suy luận logic `CryoSleep`:** Những người ngủ đông (CryoSleep) chắc chắn không thể tiêu tiền. Do đó, ta dùng `Total_Spend` để điền khuyết thiếu cho cột `CryoSleep` một cách logic thay vì điền bằng Mode/Median.
*   **Trích xuất `Family_Size` từ `Name`:** Lấy phần Họ (Surname) từ tên hành khách để đếm số người trong gia đình. Cùng một gia đình thường sẽ có chung số phận (cùng được cứu hoặc cùng bị lạc).
*   **Trích xuất `Deck` & `Side` từ `Cabin`:** Vị trí phòng ngủ (Boong nào, Mạn tàu trái/phải) ảnh hưởng trực tiếp đến khả năng sống sót khi có va chạm xảy ra.
*   **Mã hóa dữ liệu (Label Encoding):** Cuối cùng, chuyển đổi toàn bộ dữ liệu dạng chuỗi (Categorical) sang số nguyên để các mô hình dạng Cây (Tree-based) có thể tính toán được.

In [45]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
test_passenger_ids = test_df['PassengerId']

def preprocess_data(df):
    df = df.copy()
    
    # 1. Tính tổng chi tiêu và tạo cờ Spent_Money
    amenities = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in amenities:
        df[col] = df[col].fillna(0)
    df['Total_Spend'] = df[amenities].sum(axis=1)
    df['Spent_Money'] = (df['Total_Spend'] > 0).astype(int) # Feature mới cực mạnh
    
    # 2. Xử lý logic CryoSleep chặt chẽ hơn
    # Nếu có tiêu tiền -> Không thể ngủ đông. Nếu không tiêu đồng nào -> Khả năng cao đang ngủ đông.
    df['CryoSleep'] = df.apply(
        lambda row: False if pd.isna(row['CryoSleep']) and row['Total_Spend'] > 0 else
                    True if pd.isna(row['CryoSleep']) and row['Total_Spend'] == 0 else
                    row['CryoSleep'], axis=1
    )
    
    # 3. Trích xuất họ (Surname) từ Name để tìm Quy mô gia đình (Family_Size)
    df['Name'] = df['Name'].fillna('Unknown Unknown')
    df['Surname'] = df['Name'].apply(lambda x: str(x).split(' ')[-1])
    family_size = df['Surname'].value_counts().to_dict()
    df['Family_Size'] = df['Surname'].map(family_size)
    df.loc[df['Surname'] == 'Unknown', 'Family_Size'] = 1 # Nhóm không rõ tên coi như đi lẻ
    
    # 4. Xử lý cột Cabin
    df['Cabin'] = df['Cabin'].fillna('U/9999/U').astype(str) # U = Unknown
    df['Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Side'] = df['Cabin'].apply(lambda x: x.split('/')[2])
    
    # 5. Xử lý PassengerId: Tính Group Size
    df['Group'] = df['PassengerId'].apply(lambda x: str(x).split('_')[0])
    group_size = df['Group'].value_counts().to_dict()
    df['Group_Size'] = df['Group'].map(group_size)
    
    # 6. Điền Missing values cho các cột còn lại
    df['HomePlanet'] = df['HomePlanet'].fillna(df['HomePlanet'].mode()[0])
    df['Destination'] = df['Destination'].fillna(df['Destination'].mode()[0])
    df['VIP'] = df['VIP'].fillna(False)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    
    # 7. Drop các cột dạng text không cần thiết
    df.drop(['PassengerId', 'Name', 'Cabin', 'Group', 'Surname'], axis=1, inplace=True)
    
    # 8. Label Encoding
    cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
    for col in cat_features:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        
    return df

train_processed = preprocess_data(train_df)
test_processed = preprocess_data(test_df)

X = train_processed.drop('Transported', axis=1)
y = train_processed['Transported'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Kích thước tập train:", X_train.shape)
print("Kích thước tập validation:", X_val.shape)

Kích thước tập train: (6954, 16)
Kích thước tập validation: (1739, 16)


# 2. Xây dựng mô hình cơ sở (Base Models) với Ensemble Learning

Thay vì dùng các thuật toán cơ bản như Logistic Regression hay Decision Tree, bài toán này yêu cầu độ chính xác cao (>80%), do đó ta áp dụng **Ensemble Learning** - kỹ thuật kết hợp nhiều mô hình lại với nhau.

**Giải thích kỹ thuật các mô hình được chọn:**
1.  **Bagging (Random Forest):** Xây dựng hàng trăm cây quyết định (decision trees) độc lập trên các tập con dữ liệu được lấy mẫu ngẫu nhiên. Kết quả cuối cùng là trung bình của tất cả các cây. Kỹ thuật này giúp mô hình ổn định và cực kỳ chống nhiễu.
2.  **Boosting (XGBoost & LightGBM):** Khác với Bagging, Boosting xây dựng các cây tuần tự, cây sau cố gắng sửa lỗi cho cây trước. Kỹ thuật này hội tụ nhanh và thường cho điểm số cao nhất trong các cuộc thi Kaggle.

**Kiểm soát Overfitting (Học vẹt):** 
Với các đặc trưng mới được tạo ra (Family_Size, Spent_Money) rất mạnh, mô hình rất dễ bị Overfitting (điểm tập train cao nhưng test thấp). Để khắc phục, kỹ thuật tinh chỉnh siêu tham số (Hyperparameter Tuning) được áp dụng:
*   Giới hạn `max_depth` (độ sâu của cây) ở mức 5-12.
*   Giảm `learning_rate` xuống mức nhỏ (0.03 - 0.05) để mô hình học chậm lại, tăng tính tổng quát hóa (Generalization).

In [46]:
rf_model = RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_split=5, random_state=42)
xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss')
lgb_model = LGBMClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)

models = {
    'Random Forest (Bagging)': rf_model,
    'XGBoost (Boosting)': xgb_model,
    'LightGBM (Boosting)': lgb_model
}

print("--- BENCHMARK CÁC MÔ HÌNH ENSEMBLE TỐI ƯU ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"{name}: Accuracy = {acc:.4f}")

--- BENCHMARK CÁC MÔ HÌNH ENSEMBLE TỐI ƯU ---
Random Forest (Bagging): Accuracy = 0.7878
XGBoost (Boosting): Accuracy = 0.8028
LightGBM (Boosting): Accuracy = 0.8045


# 3. Kỹ thuật Stacking / Voting (Tối ưu hóa điểm số)

Dù XGBoost hay LightGBM có mạnh đến đâu, chúng vẫn có những "điểm mù" nhất định trên tập dữ liệu. Để đảm bảo mô hình có thể vượt ngưỡng 80% trên Leaderboard một cách an toàn nhất, ta sử dụng kỹ thuật **Voting Classifier**.

**Giải thích kỹ thuật Soft Voting:**
*   Thay vì lấy biểu quyết đa số (Hard Voting - ví dụ 2 mô hình bảo sống, 1 mô hình bảo chết -> kết luận sống).
*   Ta dùng **Soft Voting**: Tính trung bình cộng xác suất (probability) dự đoán của cả 3 mô hình (Random Forest, XGBoost, LightGBM) cho từng hành khách.
*   **Tại sao lại hiệu quả?** Việc trung bình hóa xác suất giúp bù trừ sai số cho nhau. Nếu XGBoost dự đoán sai một mẫu khó, Random Forest và LightGBM có thể "kéo" điểm xác suất lại, giúp kết quả dự đoán cuối cùng mượt mà và chính xác hơn rất nhiều so với bất kỳ mô hình đơn lẻ nào.

In [47]:
voting_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model),
        ('lgb', lgb_model)
    ],
    voting='soft' 
)

voting_model.fit(X_train, y_train)
voting_pred = voting_model.predict(X_val)
voting_acc = accuracy_score(y_val, voting_pred)

print(f"Voting Classifier (Bagging + Boosting): Accuracy = {voting_acc:.4f}")

Voting Classifier (Bagging + Boosting): Accuracy = 0.8010


# 4. Huấn luyện toàn bộ dữ liệu và Dự đoán

**Giải thích kỹ thuật (Retraining on Full Data):**
Trong các bước kiểm thử trước, ta đã phải cắt ra 20% dữ liệu gốc (`X_val`) để đo lường điểm Benchmark. Tuy nhiên, dữ liệu trong Machine Learning là "vàng". 
Trước khi dự đoán tập Test thực tế của Kaggle, kỹ thuật tốt nhất là phải đưa mô hình tốt nhất (Voting Classifier) đi học lại (fit) trên **toàn bộ 100% dữ liệu Train**. 
Việc được học thêm 20% dữ liệu này sẽ giúp mô hình nắm bắt được đầy đủ các patterns (quy luật) hơn, thường giúp điểm Leaderboard tăng thêm từ 0.5% đến 1%.

In [48]:
final_model = voting_model
final_model.fit(X, y)

test_predictions = final_model.predict(test_processed)

test_predictions_bool = test_predictions.astype(bool)

submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': test_predictions_bool
})

submission.to_csv('submission.csv', index=False)
print("Đã lưu file submission.csv thành công! Sẵn sàng submit lên Kaggle.")
submission.head()

Đã lưu file submission.csv thành công! Sẵn sàng submit lên Kaggle.


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
